# Umid LLM — Colab / Kaggle'da o'zbekcha til modeli yasash

Bu notebook [GPT_UZ](https://github.com/ABDULHODIY1/GPT_UZ) kodidan foydalanib, GPU'da noldan o'zbekcha LLM o'qitadi:

1. **Ma'lumot.** O'zbekcha Vikipediya va yangiliklar (Hugging Face) hamda loyihadagi savol-javoblar.
2. **Tokenizer va tokenlash** (`prepare_data.py`).
3. **Pretraining** (`train.py`). Standart algoritm — **Tanlov xotirasi** (`umid/tanlov.py`): model eng muhim K ta
   tokenni o'zi tanlab saqlaydi, xotira hajmi o'zgarmas, softmax attention yo'q.
4. **Savol-javob fine-tuning** (`sft.py`): loyihadagi savol-javoblar va o'zbekcha ko'rsatma to'plamlari.
5. **Baholash va suhbat.**

**Boshlashdan oldin:**
- Colab: *Runtime → Change runtime type → GPU (T4)*.
- Kaggle: *Settings → Accelerator → GPU*, *Internet → On*.

Sessiya uzilsa, xavotir olmang: checkpoint'lar Google Drive'ga (Colab) yoki `/kaggle/working` ga (Kaggle) yoziladi.
O'qitish katagini qayta ishga tushirsangiz, to'xtagan joydan davom etadi (`--resume`).

In [ ]:
!nvidia-smi

In [ ]:
import os
if not os.path.exists("prepare_data.py"):   # katak qayta ishga tushirilsa ham to'g'ri papkada qoladi
    if not os.path.exists("GPT_UZ"):
        !git clone -q https://github.com/ABDULHODIY1/GPT_UZ.git
    %cd GPT_UZ
!git pull -q
!pip install -q datasets

In [ ]:
# Natijalar qayerga saqlanadi (sessiya uzilsa ham yo'qolmasligi uchun)
import os
IN_KAGGLE = os.path.exists("/kaggle/working")
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = "/content/drive/MyDrive/umid"
elif IN_KAGGLE:
    SAVE = "/kaggle/working/umid"
else:
    SAVE = "checkpoints/umid_gpu"
os.makedirs(SAVE, exist_ok=True)
print("Saqlash joyi:", SAVE)

## Sozlamalar

| sozlama | ma'nosi |
|---|---|
| `MIXER` | `"tanlov"` — yangi algoritm (tavsiya); `"delta"` — eski Umid (Gated Delta) |
| `PRESET` | `tiny` (~7.6M parametr), `small` (~17.7M), `base` (~104M) — vocab 16 000 da |
| `MAX_MB` | qancha matn yuklanadi (MB). Tez sinov uchun 50, jiddiy o'qitish uchun 600+ |
| `STEPS` | pretraining qadamlari. Har qadamda `BATCH × SEQ_LEN` token |

Qancha vaqt ketishini aniq ayta olmayman, chunki u GPU va sozlamalarga bog'liq. O'qitish jurnalida
`token/s` chiqadi: `STEPS × BATCH × SEQ_LEN / token_s` sekund ketadi. Boshlash uchun `small` va 20 000 qadam
yaxshi muvozanat.

In [ ]:
MIXER   = "tanlov"   # "tanlov" (yangi) yoki "delta" (eski Umid)
PRESET  = "small"    # tiny | small | base
SEQ_LEN = 512
BATCH   = 32
STEPS   = 20000
LR      = 1e-3
MAX_MB  = 600        # tez sinov: 50
VOCAB   = 16000
SFT_STEPS = 3000

## 1. Ma'lumot: yuklash → filtrlash → tokenizer → tokenlar

- `wiki`: `wikimedia/wikipedia` (20231101.uz).
- `news`: `MLDataScientist/Uzbek_news_dataset`.
- `local`: loyihadagi `chat-data`.
- SFT uchun: `saillab/alpaca-uzbek-cleaned`, `UAzimov/uzbek-instruct-llm`.

Asosan kirillcha va juda qisqa hujjatlar tashlab yuboriladi. Har bir manba litsenziyasini uning sahifasida tekshiring.

In [ ]:
if not os.path.exists("data/tokens.npy"):
    !python prepare_data.py --out data --sources wiki,news,local --max-mb {MAX_MB} --vocab-size {VOCAB} --tok-sample-mb 20 --sft alpaca,instruct
else:
    print("data/tokens.npy tayyor — qayta tayyorlash uchun data/ papkasini o'chiring")

## 2. Pretraining

Har `--eval-every` qadamda val loss chiqadi va eng yaxshi model `SAVE/base/model.pt` ga saqlanadi.
Sessiya uzilsa, shu katakni qayta ishga tushiring: `--resume` oxirgi `last.pt` dan davom ettiradi.
GPU'da aralash aniqlik avtomatik yoqiladi: T4/P100'da fp16, A100/L4'da bf16.

In [ ]:
extra = "--select-no-conv" if MIXER == "tanlov" else "--persistent-heads 3"
!python train.py --tokens data/tokens.npy --tokenizer data/tokenizer.json --out {SAVE}/base --preset {PRESET} --mixer {MIXER} {extra} --reset-on-eot --seq-len {SEQ_LEN} --batch-size {BATCH} --steps {STEPS} --lr {LR} --warmup 500 --dropout 0.0 --eval-every 1000 --eval-windows 200 --sample-prompt "O'zbekiston" --resume

## 3. Savol-javob fine-tuning (SFT)

Loss faqat javob tokenlarida hisoblanadi. `--dialog-frac` qatorlarning bir qismini har qadamda yangidan
yaratiladigan suhbatlarga ajratadi: ularda model suhbatda aytilgan yangi faktni (ism, raqam, yangi so'z)
eslab qolishni o'rganadi.

In [ ]:
!python sft.py --init {SAVE}/base/model.pt --out {SAVE}/chat --qa chat-data/savol-javob,data/sft --steps {SFT_STEPS} --seq-len {SEQ_LEN} --batch-size {BATCH} --lr 3e-4 --warmup 100 --dialog-frac 0.25

## 4. Baholash

`qa_eval`: o'qitishda **bo'lmagan** 60 ta savol, kalit so'z bo'yicha tekshiriladi.

Diqqat: tashqi SFT to'plamlarida o'xshash savollar bo'lishi mumkin. Shuning uchun bu natijani README'dagi
kichik modellar bilan to'g'ridan-to'g'ri solishtirmang.

In [ ]:
!python -m benchmarks.qa_eval --ckpt {SAVE}/chat/model.pt --rep-penalty 1.0 --device cuda --tag gpu

## 5. Suhbat

In [ ]:
import torch
from umid import BPETokenizer, load_model
from umid.data import qa_prompt
from umid.model import generate

model, ck = load_model(f"{SAVE}/chat/model.pt")
model = model.cuda().eval()
tok = BPETokenizer.from_dict(ck["tokenizer"])
print(f"{model.num_params() / 1e6:.1f}M parametr")

def javob(savol, temperature=0.7, max_new_tokens=150):
    ids = torch.tensor([tok.encode(qa_prompt(savol))], device="cuda")
    stop = lambda t: "\n" in tok.decode([int(t[0, 0])])   # javob qator oxirida tugaydi
    out, _ = generate(model, ids, max_new_tokens, temperature=temperature, top_k=50, top_p=0.9,
                      repetition_penalty=1.15, eos_id=tok.eot_id, penalize_prompt=False, on_token=stop)
    return tok.decode(out[0].tolist()).split("\n")[0].replace("<|endoftext|>", "").strip()

for q in ["Salom", "Sen kimsan?", "O'zbekistonning poytaxti qaysi?", "Kitob o'qishning foydasi nima?",
          "Sog'lom bo'lish uchun nima qilish kerak?"]:
    print(f"Savol: {q}\nUmid:  {javob(q)}\n")

In [ ]:
# Interaktiv suhbat (to'xtatish: /chiqish). Colab'da ishlaydi; Kaggle'da kiritish oynasi yo'q — yuqoridagi javob() dan foydalaning.
!python chat.py --ckpt {SAVE}/chat/model.pt --device cuda

## 6. (Ixtiyoriy) Xuddi shu sharoitda Transformer bilan solishtirish

Bir xil ma'lumot, tokenizer, o'lcham va qadamlar ishlatiladi, `--arch gpt` bilan.
Natijani `qa_eval` va val loss bo'yicha solishtiring.

In [ ]:
!python train.py --arch gpt --tokens data/tokens.npy --tokenizer data/tokenizer.json --out {SAVE}/gpt_base --preset {PRESET} --reset-on-eot --seq-len {SEQ_LEN} --batch-size {BATCH} --steps {STEPS} --lr {LR} --warmup 500 --dropout 0.0 --eval-every 1000 --eval-windows 200 --resume
!python sft.py --init {SAVE}/gpt_base/model.pt --out {SAVE}/gpt_chat --qa chat-data/savol-javob,data/sft --steps {SFT_STEPS} --seq-len {SEQ_LEN} --batch-size {BATCH} --lr 3e-4 --warmup 100 --dialog-frac 0.25
!python -m benchmarks.qa_eval --ckpt {SAVE}/gpt_chat/model.pt --rep-penalty 1.0 --device cuda --tag gpu_gpt

## 7. (Ixtiyoriy) Modelni Hugging Face'ga yuklash

Avval https://huggingface.co/settings/tokens dan *write* token oling. Uni so'ralganda kiriting, kodga yozmang.

In [ ]:
from huggingface_hub import HfApi, login
login()   # token so'raladi
REPO = "SIZNING_ISMINGIZ/umid-uz"   # o'zgartiring
api = HfApi()
api.create_repo(REPO, exist_ok=True)
api.upload_file(path_or_fileobj=f"{SAVE}/chat/model.pt", path_in_repo="model.pt", repo_id=REPO)
api.upload_file(path_or_fileobj="data/tokenizer.json", path_in_repo="tokenizer.json", repo_id=REPO)
print("https://huggingface.co/" + REPO)